[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/03_Training_Techniques/02_Mixed_Precision_Training.ipynb)

# Mixed Precision Training: From Theory to Practice

This notebook explores **mixed precision training** (Micikevicius et al., 2018), which uses
lower-precision floating-point formats (float16, bfloat16) alongside float32 to reduce memory,
increase throughput, and maintain accuracy.

We will:
1. Explain why float32 is wasteful for most training operations
2. Compare number formats: float32 vs float16 vs bfloat16
3. Understand the loss scaling problem and how GradScaler solves it
4. Use `torch.autocast` — the modern PyTorch AMP API
5. Train ResNet-18 on CIFAR-10 with both float32 and mixed precision
6. Benchmark training time, memory usage, and final accuracy

**Reference:** Micikevicius, P. et al. (2018). *Mixed Precision Training.* ICLR 2018.
https://arxiv.org/abs/1710.03740

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torchvision import transforms
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from src.utils.device import set_device, set_seed, set_deterministic

device = set_device()
set_seed(42)
set_deterministic()

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. Why Mixed Precision?

Three reasons to train in lower precision:

**Compute speedup** — NVIDIA Tensor Cores perform fp16/bf16 matrix multiplications at 2-8x
the throughput of fp32. Apple's MPS also benefits from fp16.

**Memory savings** — fp16/bf16 use 2 bytes per value vs 4 for fp32. This halves activation
memory and allows larger batch sizes or longer sequences.

**Training throughput** — Faster compute + larger batches = more samples per second.

| Metric | float32 | float16/bfloat16 |
|---|---|---|
| Bytes per value | 4 | 2 |
| Memory for 11M params | 44 MB | 22 MB |
| Tensor Core eligible | No | Yes |

## 2. Number Formats: float32 vs float16 vs bfloat16

IEEE 754 floating-point: $x = (-1)^s \times 2^{e - \text{bias}} \times (1 + m)$

```
float32:  [1 sign] [8 exponent] [23 mantissa]  — high range, high precision
float16:  [1 sign] [5 exponent] [10 mantissa]  — low range, medium precision
bfloat16: [1 sign] [8 exponent] [ 7 mantissa]  — high range, low precision
```

- **float32**: Range $\pm 3.4 \times 10^{38}$, precision ~7 decimal digits
- **float16**: Range $\pm 6.5 \times 10^{4}$, precision ~3.3 digits. Small range causes gradient underflow.
- **bfloat16**: Range $\pm 3.4 \times 10^{38}$ (same as fp32!), precision ~2.4 digits

**Key insight:** bfloat16 is often preferred because it has the same exponent range as float32,
so gradients rarely overflow or underflow. The reduced mantissa precision is tolerable for SGD,
which is inherently noisy.

In [ ]:
# Inspect the range and precision of each format
for dtype, name in [(torch.float32, "float32"), (torch.float16, "float16"), (torch.bfloat16, "bfloat16")]:
    info = torch.finfo(dtype)
    print(f"{name:>8s}: range=[{info.min:.2e}, {info.max:.2e}], "
          f"smallest normal={info.tiny:.2e}, eps={info.eps:.2e}")

# Demonstrate gradient underflow in fp16
grad_val = torch.tensor(1e-6, dtype=torch.float32)
print(f"\nGradient value {grad_val.item():.1e} in float32: {grad_val.item()}")
print(f"  Cast to float16:  {grad_val.to(torch.float16).item()}")
print(f"  Cast to bfloat16: {grad_val.to(torch.bfloat16).item()}")

grad_tiny = torch.tensor(1e-8, dtype=torch.float32)
print(f"\nGradient value {grad_tiny.item():.1e} in float32: {grad_tiny.item()}")
print(f"  Cast to float16:  {grad_tiny.to(torch.float16).item()}  <- UNDERFLOW!")
print(f"  Cast to bfloat16: {grad_tiny.to(torch.bfloat16).item()}")

## 3. The Loss Scaling Problem

Many gradients in deep networks have magnitudes in the range $[10^{-8}, 10^{-5}]$, which
is below float16's smallest representable normal number ($6.1 \times 10^{-5}$). These
gradients become zero ("underflow"), and the model stops learning.

**Loss scaling** multiplies the loss by a large factor $S$ before `backward()`, which
scales all gradients by the same factor:

$\nabla_\theta (S \cdot L) = S \cdot \nabla_\theta L$

After the backward pass, gradients are divided by $S$ before the optimizer step.

**Dynamic loss scaling** (`GradScaler`):
- Starts with a large $S$ (e.g., $2^{16} = 65536$)
- If gradients contain inf/NaN → overflow detected → halve $S$, skip optimizer step
- Periodically increase $S$ to use the full fp16 range

**bfloat16 sidesteps this entirely** — its range matches float32, so gradients don't underflow.
No GradScaler needed.

In [ ]:
# Demonstrate loss scaling: a tiny gradient underflows in fp16 but is preserved with scaling
small_grad = torch.tensor(1e-8, dtype=torch.float32)
scale_factor = 2**16  # 65536

print("Without loss scaling:")
print(f"  Gradient in fp32:  {small_grad.item():.2e}")
print(f"  Gradient in fp16:  {small_grad.to(torch.float16).item():.2e}  <- underflow to zero!")

scaled_grad = small_grad * scale_factor
print(f"\nWith loss scaling (S={scale_factor}):")
print(f"  Scaled gradient in fp32: {scaled_grad.item():.2e}")
print(f"  Scaled gradient in fp16: {scaled_grad.to(torch.float16).item():.2e}  <- preserved!")
print(f"  After unscaling:         {scaled_grad.to(torch.float16).item() / scale_factor:.2e}")

## 4. torch.autocast

`torch.autocast(device_type, dtype)` automatically selects the appropriate dtype per operation:

- **Matmuls and convolutions** → run in fp16/bf16 (benefit from Tensor Cores)
- **Reductions, softmax, loss functions, layer norms** → stay in fp32 (need numerical stability)

The API is device-aware:

| Device | autocast dtype | GradScaler | Notes |
|---|---|---|---|
| CUDA | fp16 (default), bf16 | Yes (fp16), No (bf16) | Full Tensor Core support |
| MPS | fp16 | No (not supported) | Apple Silicon Metal acceleration |
| CPU | bf16 | No | Useful for testing, limited speedup |

Note: the older `torch.cuda.amp.autocast` is deprecated since PyTorch 2.4. Use the
device-generic `torch.autocast` instead.

In [ ]:
# Demonstrate what autocast does: selects dtype per operation
x = torch.randn(4, 16, device=device)
weight = torch.randn(16, 16, device=device)

# Determine autocast settings for this device
if device == 'cuda':
    autocast_device = 'cuda'
    autocast_dtype = torch.float16
elif device == 'mps':
    autocast_device = 'mps'
    autocast_dtype = torch.float16
else:
    autocast_device = 'cpu'
    autocast_dtype = torch.bfloat16

print(f"Using autocast('{autocast_device}', dtype={autocast_dtype})")

print(f"\nWithout autocast:")
out = x @ weight
print(f"  matmul output dtype: {out.dtype}")

print(f"\nWith autocast:")
with torch.autocast(device_type=autocast_device, dtype=autocast_dtype):
    out_amp = x @ weight
    print(f"  matmul output dtype: {out_amp.dtype}  <- downcast for speed")
    loss = out_amp.sum()
    print(f"  sum (reduction) dtype: {loss.dtype}")

## 5. Implementation

We'll implement two training functions and compare them:
1. **Baseline**: Standard float32 training
2. **Mixed Precision**: `torch.autocast` + `torch.GradScaler` (when needed)

First, a helper that auto-detects the best AMP config for the current device.

In [ ]:
def get_amp_config(device):
    """
    Return (device_type, dtype, use_scaler) for mixed precision training.
    
    - CUDA: bf16 if supported (Ampere+), otherwise fp16 with GradScaler
    - MPS: fp16, no GradScaler (not supported)
    - CPU: bf16, no GradScaler
    """
    device_type = device if isinstance(device, str) else device.type
    if device_type == 'cuda':
        if torch.cuda.is_bf16_supported():
            return device_type, torch.bfloat16, False
        return device_type, torch.float16, True
    elif device_type == 'mps':
        return device_type, torch.float16, False
    else:
        return device_type, torch.bfloat16, False


amp_device_type, amp_dtype, use_scaler = get_amp_config(device)
print(f"AMP config for '{device}':")
print(f"  autocast device_type: {amp_device_type}")
print(f"  autocast dtype: {amp_dtype}")
print(f"  use GradScaler: {use_scaler}")

In [ ]:
def train_model_timed(model, train_dataloader, test_dataloader, epochs, criterion,
                      optimizer, device, scheduler=None):
    """Standard float32 training with epoch timing."""
    model.to(device)
    results = {
        "train_loss_per_epoch": [], "train_acc_per_epoch": [],
        "val_loss_per_epoch": [], "val_acc_per_epoch": [], "epoch_times": [],
    }

    for epoch in tqdm(range(epochs)):
        model.train()
        training_loss, training_acc = 0.0, 0.0
        epoch_start = time.perf_counter()

        for batch, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            training_loss += loss.item()
            training_acc += (outputs.argmax(1) == labels).float().mean().item()

        if scheduler is not None:
            scheduler.step()

        epoch_time = time.perf_counter() - epoch_start
        epoch_loss_tr = training_loss / len(train_dataloader)
        epoch_acc_tr = training_acc / len(train_dataloader)

        model.eval()
        test_loss, test_acc = 0.0, 0.0
        with torch.no_grad():
            for batch, (inputs, labels) in enumerate(test_dataloader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                test_acc += (outputs.argmax(1) == labels).float().mean().item()

        epoch_loss_te = test_loss / len(test_dataloader)
        epoch_acc_te = test_acc / len(test_dataloader)

        print(f"Epoch {epoch}: train_loss={epoch_loss_tr:.4f}, train_acc={epoch_acc_tr:.4f}, "
              f"val_loss={epoch_loss_te:.4f}, val_acc={epoch_acc_te:.4f}, time={epoch_time:.2f}s")

        results["train_loss_per_epoch"].append(epoch_loss_tr)
        results["train_acc_per_epoch"].append(epoch_acc_tr)
        results["val_loss_per_epoch"].append(epoch_loss_te)
        results["val_acc_per_epoch"].append(epoch_acc_te)
        results["epoch_times"].append(epoch_time)

    return model, results


def train_model_amp(model, train_dataloader, test_dataloader, epochs, criterion,
                    optimizer, device, scheduler=None):
    """Mixed precision training with autocast + optional GradScaler."""
    model.to(device)
    amp_device_type, amp_dtype, use_scaler = get_amp_config(device)
    scaler = torch.GradScaler(device) if use_scaler else None

    results = {
        "train_loss_per_epoch": [], "train_acc_per_epoch": [],
        "val_loss_per_epoch": [], "val_acc_per_epoch": [], "epoch_times": [],
    }

    for epoch in tqdm(range(epochs)):
        model.train()
        training_loss, training_acc = 0.0, 0.0
        epoch_start = time.perf_counter()

        for batch, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # --- Forward pass under autocast ---
            with torch.autocast(device_type=amp_device_type, dtype=amp_dtype):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            # --- Backward with optional loss scaling ---
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            training_loss += loss.item()
            training_acc += (outputs.argmax(1) == labels).float().mean().item()

        if scheduler is not None:
            scheduler.step()

        epoch_time = time.perf_counter() - epoch_start
        epoch_loss_tr = training_loss / len(train_dataloader)
        epoch_acc_tr = training_acc / len(train_dataloader)

        model.eval()
        test_loss, test_acc = 0.0, 0.0
        with torch.no_grad():
            with torch.autocast(device_type=amp_device_type, dtype=amp_dtype):
                for batch, (inputs, labels) in enumerate(test_dataloader):
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    test_loss += loss.item()
                    test_acc += (outputs.argmax(1) == labels).float().mean().item()

        epoch_loss_te = test_loss / len(test_dataloader)
        epoch_acc_te = test_acc / len(test_dataloader)

        print(f"Epoch {epoch}: train_loss={epoch_loss_tr:.4f}, train_acc={epoch_acc_tr:.4f}, "
              f"val_loss={epoch_loss_te:.4f}, val_acc={epoch_acc_te:.4f}, time={epoch_time:.2f}s")

        results["train_loss_per_epoch"].append(epoch_loss_tr)
        results["train_acc_per_epoch"].append(epoch_acc_tr)
        results["val_loss_per_epoch"].append(epoch_loss_te)
        results["val_acc_per_epoch"].append(epoch_acc_te)
        results["epoch_times"].append(epoch_time)

    return model, results


print("Training functions ready.")

## 6. Benchmark Setup

We'll train **ResNet-18** on **CIFAR-10** for 5 epochs, comparing:
- Standard float32 training
- Mixed precision training (autocast + GradScaler when applicable)

Both runs use the same random seed for fair comparison.

In [ ]:
from src.data.loaders import build_dataloaders_cifar

BATCH_SIZE = 128
NUM_WORKERS = 4
NUM_EPOCHS = 5
LEARNING_RATE = 0.001

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_dl, test_dl, test_viz, class_names = build_dataloaders_cifar(
    data_path='../../data',
    train_transforms=train_transforms,
    test_transforms=test_transforms,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print(f"Train batches: {len(train_dl)}, Test batches: {len(test_dl)}")
print(f"Classes: {class_names}")

In [ ]:
def make_resnet18_cifar10():
    """Create a ResNet-18 adapted for CIFAR-10 (32x32 images, 10 classes)."""
    model = models.resnet18(weights=None)
    # Adapt for 32x32: smaller first conv, remove aggressive maxpool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


# Verify and count parameters
test_model = make_resnet18_cifar10().to(device)
test_out = test_model(torch.randn(2, 3, 32, 32, device=device))
print(f"Output shape: {test_out.shape}")

num_params = sum(p.numel() for p in test_model.parameters())
print(f"Parameters: {num_params:,}")
print(f"  float32 size: {num_params * 4 / 1024**2:.1f} MB")
print(f"  float16 size: {num_params * 2 / 1024**2:.1f} MB")
del test_model, test_out

In [ ]:
print("=" * 60)
print("Baseline: float32 training")
print("=" * 60)
set_seed(42)
model_fp32 = make_resnet18_cifar10()
optimizer_fp32 = torch.optim.Adam(model_fp32.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

model_fp32, results_fp32 = train_model_timed(
    model_fp32, train_dl, test_dl, NUM_EPOCHS, criterion, optimizer_fp32, device
)

In [ ]:
print("=" * 60)
print("Mixed Precision: AMP training")
print("=" * 60)
set_seed(42)
model_amp = make_resnet18_cifar10()
optimizer_amp = torch.optim.Adam(model_amp.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

model_amp, results_amp = train_model_amp(
    model_amp, train_dl, test_dl, NUM_EPOCHS, criterion, optimizer_amp, device
)

## 7. Results

In [ ]:
fp32_total = sum(results_fp32['epoch_times'])
amp_total = sum(results_amp['epoch_times'])

summary = pd.DataFrame({
    'Metric': ['Total training time (s)', 'Avg epoch time (s)',
               'Final val accuracy', 'Final val loss', 'Speedup'],
    'float32': [f"{fp32_total:.1f}", f"{fp32_total/NUM_EPOCHS:.1f}",
                f"{results_fp32['val_acc_per_epoch'][-1]:.4f}",
                f"{results_fp32['val_loss_per_epoch'][-1]:.4f}",
                "1.00x (baseline)"],
    'Mixed Precision': [f"{amp_total:.1f}", f"{amp_total/NUM_EPOCHS:.1f}",
                        f"{results_amp['val_acc_per_epoch'][-1]:.4f}",
                        f"{results_amp['val_loss_per_epoch'][-1]:.4f}",
                        f"{fp32_total/amp_total:.2f}x"],
})
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Epoch training times
ax = axes[0]
x = np.arange(NUM_EPOCHS)
width = 0.35
ax.bar(x - width/2, results_fp32['epoch_times'], width, label='float32', color='tab:blue', alpha=0.7)
ax.bar(x + width/2, results_amp['epoch_times'], width, label='Mixed Precision', color='tab:orange', alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Training Time per Epoch', fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Validation accuracy
ax = axes[1]
ax.plot(range(NUM_EPOCHS), results_fp32['val_acc_per_epoch'], 'o-', label='float32', linewidth=2)
ax.plot(range(NUM_EPOCHS), results_amp['val_acc_per_epoch'], 's-', label='Mixed Precision', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Accuracy Convergence', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Training loss
ax = axes[2]
ax.plot(range(NUM_EPOCHS), results_fp32['train_loss_per_epoch'], 'o-', label='float32', linewidth=2)
ax.plot(range(NUM_EPOCHS), results_amp['train_loss_per_epoch'], 's-', label='Mixed Precision', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Loss Convergence', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Memory Analysis

ResNet-18 has ~11.2M parameters:
- float32: $11.2\text{M} \times 4\text{B} = 44.7\text{ MB}$ for parameters
- float16: $11.2\text{M} \times 2\text{B} = 22.4\text{ MB}$ for parameters

With Adam optimizer (stores 2 extra tensors per parameter):
- float32 total: $44.7 \times 3 \approx 134\text{ MB}$
- Mixed precision: master weights in fp32 ($44.7$) + optimizer states in fp32 ($89.4$) + activations in fp16 → significant activation memory savings

In [ ]:
def measure_model_memory(model, name):
    total_bytes = sum(p.nelement() * p.element_size() for p in model.parameters())
    total_mb = total_bytes / 1024**2
    print(f"{name}: {total_mb:.1f} MB ({sum(p.numel() for p in model.parameters()):,} params)")
    return total_mb

print("Parameter memory:")
mem_fp32 = measure_model_memory(model_fp32, "float32")

model_fp16_demo = make_resnet18_cifar10().half()
mem_fp16 = measure_model_memory(model_fp16_demo, "float16")
del model_fp16_demo

print(f"\nMemory savings: {mem_fp32 - mem_fp16:.1f} MB ({(1 - mem_fp16/mem_fp32)*100:.0f}%)")

if device == 'cuda':
    print(f"\nCUDA peak memory: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")

## 9. Reusable Implementation

Both `train_model_amp` and `get_amp_config` are available in `src/` for reuse:

```python
from src.training.trainer import train_model_amp
from src.utils.device import get_amp_config
```

In [ ]:
from src.training.trainer import train_model_amp as amp_fn
from src.utils.device import get_amp_config as amp_cfg

dt, dtype, scaler = amp_cfg(device)
print(f"get_amp_config('{device}'): device_type={dt}, dtype={dtype}, use_scaler={scaler}")
print("All src/ imports verified.")

## 10. Key Takeaways

1. **Mixed precision halves memory for activations and parameters.** Float16/bfloat16 use 2 bytes vs 4, enabling larger batch sizes or models.

2. **`torch.autocast` selects dtypes per operation.** Matrix multiplications and convolutions run in fp16/bf16 for speed; reductions and normalizations stay in fp32 for stability.

3. **Loss scaling prevents gradient underflow in fp16.** `torch.GradScaler` dynamically adjusts the scale factor. bfloat16 doesn't need it (same range as fp32).

4. **bfloat16 is usually preferred over fp16.** Same exponent range as fp32 means no loss scaling needed. Available on CUDA (Ampere+) and CPU.

5. **Accuracy is equivalent.** Mixed precision produces the same final accuracy as float32 — the precision loss is compensated by SGD's inherent noise.

6. **Device compatibility:**

| | CUDA | MPS | CPU |
|---|---|---|---|
| fp16 autocast | Yes | Yes | No |
| bf16 autocast | Yes (Ampere+) | No | Yes |
| GradScaler | Yes (fp16 only) | No | No |

### Further Reading

- Micikevicius et al. (2018). *Mixed Precision Training.* https://arxiv.org/abs/1710.03740
- PyTorch AMP docs: https://pytorch.org/docs/stable/amp.html
- NVIDIA Mixed Precision Training: https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/

## 11. Run on GPU (Modal)

Run the mixed precision benchmark on a remote NVIDIA GPU via [Modal](https://modal.com).
On CUDA with Ampere+ GPUs, mixed precision shows 1.5-2x speedup with Tensor Cores.

```bash
# One-time setup:
pip install modal
modal token set
```

In [ ]:
from src.infra.modal_runner import run_training

# Float32 baseline on A100
results_gpu_fp32 = run_training.remote(
    model_factory_name="make_resnet18_cifar10",
    trainer_name="train_model",
    dataset="cifar10",
    epochs=5,
    batch_size=128,
    lr=0.001,
)

# Mixed precision on A100
results_gpu_amp = run_training.remote(
    model_factory_name="make_resnet18_cifar10",
    trainer_name="train_model_amp",
    dataset="cifar10",
    epochs=5,
    batch_size=128,
    lr=0.001,
)

print(f"GPU: {results_gpu_fp32['gpu_name']}")
print(f"Peak memory FP32: {results_gpu_fp32['peak_memory_mb']:.0f} MB")
print(f"Peak memory AMP:  {results_gpu_amp['peak_memory_mb']:.0f} MB")
print()

fp32_time = sum(results_gpu_fp32["epoch_times"])
amp_time = sum(results_gpu_amp["epoch_times"])
print(f"FP32: {fp32_time:.1f}s total, val_acc={results_gpu_fp32['val_acc_per_epoch'][-1]:.4f}")
print(f"AMP:  {amp_time:.1f}s total, val_acc={results_gpu_amp['val_acc_per_epoch'][-1]:.4f}")
print(f"Speedup: {fp32_time/amp_time:.2f}x")